# Python for Simulation, Bioinformatics, and Scientific Workflows

This notebook scaffold mirrors the Python-first workflow: biological simulation, sequence summary, metadata validation, and reproducibility documentation.

In [ ]:
from pathlib import Path
import pandas as pd

article_dir = Path.cwd().parent
parameters = pd.read_csv(article_dir / 'data' / 'simulation_parameters.csv')
parameters.head()

In [ ]:
def simulate_logistic(initial_population, growth_rate, carrying_capacity, dt, steps):
    population = float(initial_population)
    rows = []
    for step in range(steps + 1):
        rows.append({'step': step, 'time': step * dt, 'population': population})
        growth = growth_rate * population * (1 - population / carrying_capacity)
        population = max(population + dt * growth, 0.0)
    return pd.DataFrame(rows)

trajectory = simulate_logistic(25, 0.35, 1000, 0.1, 200)
trajectory.tail().round(5)

In [ ]:
fasta_text = (article_dir / 'data' / 'sequences.fasta').read_text()
records = {}
current = None
seq = []
for line in fasta_text.strip().splitlines():
    if line.startswith('>'):
        if current is not None:
            records[current] = ''.join(seq).upper()
        current = line[1:].split()[0]
        seq = []
    else:
        seq.append(line.strip())
if current is not None:
    records[current] = ''.join(seq).upper()

def gc(seq):
    valid = [b for b in seq if b in set('ACGT')]
    return (valid.count('G') + valid.count('C')) / len(valid)

pd.DataFrame({'sequence_id': list(records.keys()), 'length': [len(s) for s in records.values()], 'gc_content': [gc(s) for s in records.values()]}).round(5)